In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
import os

# Bootstrap function

In [2]:
def bootstrap_spearman(x, y, nboot=100, conf=0.95, random_state=None):

    # initiate random number generator
    rng = np.random.default_rng(random_state)
    n = len(x)
    rhos = []

    # calculate rho for each bootstrapped sample
    for i in range(nboot):
        idx = rng.integers(0, n, n)   # sample indices with replacement
        rho, _ = spearmanr(x[idx], y[idx])
        if not np.isnan(rho):
            rhos.append(rho)

    # get rho for the sample
    rho_hat, _ = spearmanr(x, y)

    # calculate confidence intervals
    alpha = (1 - conf) / 2
    ci = np.quantile(rhos, [alpha, 1 - alpha])
    #return {"rho": rho_hat, "ci": ci, "boot_rhos": rhos, "ci_low" : ci[0], "ci_high" : ci[1]}
    return [rho_hat, ci[0], ci[1]]

### Example

In [3]:
dx = np.array([1.2, 2.5, 3.0, 4.1, 5.3])  # your data (n=5)
dy = np.array([2.1, 2.8, 0.9, 4.0, 2.8])

In [4]:
dboot = bootstrap_spearman(dx, dy, nboot=50, conf=0.95, random_state=15)
#dboot['rho'], dboot['ci_low'], dboot['ci_high']
dboot

[0.46169025843831935, -0.6666666666666666, 1.0]

# DATA

In [5]:
df = pd.read_csv("18_garden_clusters.tsv", sep = "\t", header=0)
df.head()

,sample,SITE_ID,garden,offset_rda,marker,trait,train_group,Trait_name,POP_SITE,POP_ID,SET,mean,sd,n,group
0,2209_PR,PR,PR,0.340529,1000,Height,Central,Height,2209_PR,2209,TEST,980.525258,302.158899,4.0,West
1,325_PR,PR,PR,0.378512,1000,Height,Central,Height,325_PR,325,TEST,1239.891145,121.743290,8.0,Central
2,4351_PR,PR,PR,0.278826,1000,Height,Central,Height,4351_PR,4351,TEST,1423.598605,68.068593,4.0,Central
3,4420_PR,PR,PR,0.447724,1000,Height,Central,Height,4420_PR,4420,TEST,678.153897,154.272486,7.0,West
4,6805_PR,PR,PR,0.339265,1000,Height,Central,Height,6805_PR,6805,TEST,1353.974749,229.855027,7.0,East


In [6]:
df[['train_group','group']].drop_duplicates()

,train_group,group
0,Central,West
1,Central,Central
4,Central,East
26,West,West
27,West,Central
30,West,East
52,East,East
59,East,Central
88,East,West


In [7]:
df[df['Trait_name'].isna()]

,sample,SITE_ID,garden,offset_rda,marker,trait,train_group,Trait_name,POP_SITE,POP_ID,SET,mean,sd,n,group


# Bootstrapping

### Calculating rho and running bootstrap for samples n>=5 and ignoring cases when all offsets are = 0

In [8]:
df["marker"].drop_duplicates().values

array([1000])

In [9]:
import warnings
warnings.filterwarnings("ignore", message="An input array is constant")

nboots = 1000
i=0
results = []
for trait in df["Trait_name"].drop_duplicates().values:
    print(trait)
    for marker in df["marker"].drop_duplicates().values:
        #print(marker)
        for dset in ["TRAIN", "TEST"]:
            #print(dset)
            for site in ['AC','ML','CH','PR']:
                #print(site)
                for train_group in ["West","Central","East"]:
                    df_ = df[(df['SET']==dset) & (df['marker']==marker) & (df['Trait_name']==trait) & (df["SITE_ID"]==site) & (df["train_group"]==train_group)].reset_index(drop=True)
                    size = df_.shape[0]
                    offset_sum = df_['offset_rda'].sum()
                    #group = df_['group'].first()
                    if (size > 4) & (offset_sum > 0):
                        rho, ci_low, ci_high = bootstrap_spearman(df_['offset_rda'], df_['mean'], nboot=nboots, conf=0.95, random_state=123+i)
                    else:
                        rho, ci_low, ci_high = np.nan, np.nan, np.nan
                    results.append((dset, marker, trait, site, "combined", train_group, rho, ci_low, ci_high, size, nboots))
                    
                    i+=1
dR = pd.DataFrame(results, columns = ["SET","marker","trait","SITE_ID","group","train_group","rho","CI_low","CI_high","n_samples","n_boot"])

Height
Biomass_Increment
Biomass_Increment_1980
Biomass_Increment_1985
Biomass_Increment_1990
Biomass_Increment_1995
Biomass_Increment_2000
Biomass_Increment_2005
Biomass_Increment_2010
Biomass_Increment_2015
Average_Ring_Density
DBH
Rs
Rl
Rr
Rc


In [10]:
dR.head(20)

,SET,marker,trait,SITE_ID,group,train_group,rho,CI_low,CI_high,n_samples,n_boot
0,TRAIN,1000,Height,AC,combined,West,NaN,NaN,NaN,0,1000
1,TRAIN,1000,Height,AC,combined,Central,-0.200000,-1.000000,1.000000,5,1000
2,TRAIN,1000,Height,AC,combined,East,NaN,NaN,NaN,0,1000
3,TRAIN,1000,Height,ML,combined,West,NaN,NaN,NaN,0,1000
4,TRAIN,1000,Height,ML,combined,Central,-0.064912,-0.480167,0.411666,19,1000
5,TRAIN,1000,Height,ML,combined,East,-0.436533,-0.736310,0.065833,18,1000
6,TRAIN,1000,Height,CH,combined,West,-0.500000,-1.000000,1.000000,5,1000
7,TRAIN,1000,Height,CH,combined,Central,-0.382353,-0.675773,0.120206,17,1000
8,TRAIN,1000,Height,CH,combined,East,-0.709890,-0.928152,-0.299755,14,1000
9,TRAIN,1000,Height,PR,combined,West,-0.252747,-0.779980,0.455832,14,1000


In [11]:
results
dR = pd.DataFrame(results, columns = ["SET","marker","trait","SITE_ID","group","train_group","rho","CI_low","CI_high","n_samples","n_boot"])
dR.head()

,SET,marker,trait,SITE_ID,group,train_group,rho,CI_low,CI_high,n_samples,n_boot
0,TRAIN,1000,Height,AC,combined,West,NaN,NaN,NaN,0,1000
1,TRAIN,1000,Height,AC,combined,Central,-0.200000,-1.000000,1.000000,5,1000
2,TRAIN,1000,Height,AC,combined,East,NaN,NaN,NaN,0,1000
3,TRAIN,1000,Height,ML,combined,West,NaN,NaN,NaN,0,1000
4,TRAIN,1000,Height,ML,combined,Central,-0.064912,-0.480167,0.411666,19,1000


In [12]:
dR.to_csv("19_bootstrap.tsv", sep="\t", header=True, index=False)

In [17]:
#df_ = df[(df['SET']=="TEST") & (df['marker']=='100LF') & (df['Trait_name']=='Biomass_Increment_1980') & (df["SITE_ID"]=='AC')].reset_index(drop=True)
#size = df_.shape[0]
#df_

In [18]:
df_['offset'].sum()

0.0